# Pytorch tutorial - Crash course

https://www.youtube.com/watch?v=uq7sbUlIDR8&t=34s

In [1]:
import torch

In [2]:
import numpy as np # works only on the CPU unlike Tensors that can go on the GPU

In [3]:
torch.cuda.is_available()

False

In [4]:
print(np.__version__)
print(torch.__version__)

1.26.4
2.12.0


## Tensors

Works the same as in `TensorFlow`

In [5]:
my_array = np.array([[4,5,6], [7,8,9]])

In [6]:
tensor = torch.Tensor([[1,2,3],[4,5,6]]) # works exactly like numpy = replacement of numpy but for GPU

In [7]:
tensor

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [8]:
tensor2 = torch.from_numpy(my_array)

In [9]:
tensor2

tensor([[4, 5, 6],
        [7, 8, 9]])

Exactly like Numpy I have a lot of functions to create tensors

- arr.shape -> tensor.shape
- arr.dtype -> tensor.dtype
- arr.device -> tensor.device

In [10]:
my_array.shape, tensor.shape

((2, 3), torch.Size([2, 3]))

In [11]:
my_array.dtype, tensor.dtype

(dtype('int64'), torch.float32)

In [12]:
tensor.device

device(type='cpu')

In [13]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
tensor = tensor.to(device=device) # en Pytorch, tout ce qui est déplacé crée un nouvel objet donc à réassigner

In [14]:
tensor.device

device(type='mps', index=0)

In [15]:
tensor.cpu().numpy() # converstion vers le CPU pour obtenir numpy

array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)

In [16]:
tensor.cpu() # before using it with numpy

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [17]:
a = torch.tensor([2.,3.], requires_grad=True) # this says to Pytorch please keep the information on the graph
b = torch.tensor([6., 4.], requires_grad=True)

f = 3*a**3-b**2

In [18]:
f

tensor([-12.,  65.], grad_fn=<SubBackward0>)

In [19]:
f.backward(gradient=torch.tensor([1,1]))

In [20]:
import torch.nn as nn # build the layers
import torch.nn.functional as F # the activations functions
import torch.optim as optim # the optimizers
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [21]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [36]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

In [38]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [39]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [46]:
class BCNet(nn.Module):
    # this is the architecture only
    def __init__(self):
        super().__init__()
        self.fcl1 = nn.Linear(30, 64) # 30 inputs (from data) and 64 outpurts (choice)
        self.fcl2 = nn.Linear(64, 32)
        self.fcl3 = nn.Linear(32, 1)
    # we need to explain what happens when we put data inside -> forwars

    def forward(self, x):
        x = F.relu(self.fcl1(x))
        x = F.relu(self.fcl2(x))
        x = F.sigmoid(self.fcl3(x))

        return x

In [47]:
# criterion and optimizer
model = BCNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(lr=0.001, params=model.parameters())

In [48]:
model.to("mps")

BCNet(
  (fcl1): Linear(in_features=30, out_features=64, bias=True)
  (fcl2): Linear(in_features=64, out_features=32, bias=True)
  (fcl3): Linear(in_features=32, out_features=1, bias=True)
)

In [50]:
# Here we need to manualluy explain how to train (same for tensorflow without keras)
epochs = 20
for epoch in range(epochs):
    # set the model in training mode
    model.train()
    running_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to("mps"), y_batch.to("mps")
        # reset all the gradients calculations
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss
    print(f"epoch {epoch+1} - loss: {running_loss/len(train_loader)}")

epoch 1 - loss: 0.026339981704950333
epoch 2 - loss: 0.02542838640511036
epoch 3 - loss: 0.023438861593604088
epoch 4 - loss: 0.02238600142300129
epoch 5 - loss: 0.021034667268395424
epoch 6 - loss: 0.02061941660940647
epoch 7 - loss: 0.01877768151462078
epoch 8 - loss: 0.01928931474685669
epoch 9 - loss: 0.017474226653575897
epoch 10 - loss: 0.01588885299861431
epoch 11 - loss: 0.01504701841622591
epoch 12 - loss: 0.014211016707122326
epoch 13 - loss: 0.013297516852617264
epoch 14 - loss: 0.016175540164113045
epoch 15 - loss: 0.02705312892794609
epoch 16 - loss: 0.011817057617008686
epoch 17 - loss: 0.00996934249997139
epoch 18 - loss: 0.009604019112884998
epoch 19 - loss: 0.011439740657806396
epoch 20 - loss: 0.008135217241942883


In [54]:
# to evaluate the model (we do not need the gradient)
with torch.no_grad():
    # put the model into eval mode
    model.eval()

    preds = model(X_test_scaled_tensor.to("mps"))
    loss = criterion(preds, y_test_tensor.to("mps")).item()

    accuracy = ((preds >= 0.5) == y_test_tensor.to("mps")).float().mean().item()

In [55]:
accuracy

0.9649122953414917